### Libraries and Memory Optimization Function

In [17]:
import pandas as pd
import numpy as np
import gc

# Setting pandas to show all columns during inspection
pd.set_option('display.max_columns', 100)

def reduce_mem_usage(df):
    """Iterates through all columns of a dataframe and modifies the data type to reduce memory usage."""
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage of dataframe is {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        # Check if column is numeric using pandas' built-in function
        if pd.api.types.is_numeric_dtype(col_type):
            try:
                c_min = df[col].min()
                c_max = df[col].max()
                
                if pd.api.types.is_integer_dtype(col_type):
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                elif pd.api.types.is_float_dtype(col_type):
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
            except (TypeError, OverflowError):
                # Skip columns that can't be compared numerically
                pass
        elif col_type == object:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage after optimization is: {end_mem:.2f} MB')
    print(f'Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%')
    
    return df

### Loading and Filtering Data (Testing Scope)

In [18]:
RAW_PATH = "../data/raw"

print("Loading calendar and prices...")
calendar = pd.read_csv(f'{RAW_PATH}/calendar.csv')
prices = pd.read_csv(f'{RAW_PATH}/sell_prices.csv')

print("Loading sales (Evaluation)...")
sales = pd.read_csv(f'{RAW_PATH}/sales_train_evaluation.csv')

# FILTERING FOR TESTING: Keeping only one store to test the logic quickly
# Remove this block later when running the full dataset
store_to_test = 'CA_1'
sales = sales[sales['store_id'] == store_to_test].copy()
prices = prices[prices['store_id'] == store_to_test].copy()

# Applying memory reduction
calendar = reduce_mem_usage(calendar)
prices = reduce_mem_usage(prices)
sales = reduce_mem_usage(sales)

print(f"\nFiltered sales shape for {store_to_test}: {sales.shape}")

Loading calendar and prices...
Loading sales (Evaluation)...
Memory usage of dataframe is 0.21 MB
Memory usage after optimization is: 0.12 MB
Decreased by 41.9%
Memory usage of dataframe is 21.31 MB
Memory usage after optimization is: 13.32 MB
Decreased by 37.5%
Memory usage of dataframe is 45.29 MB
Memory usage after optimization is: 6.58 MB
Decreased by 85.5%

Filtered sales shape for CA_1: (3049, 1947)


### Melting the Sales Data (Wide to Long)

In [19]:
# Transforming the day columns (d_1, d_2...) into rows
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

print("Melting data...")
df = pd.melt(sales, id_vars=id_vars, var_name='d', value_name='sales')

# Freeing up memory
del sales
gc.collect()

display(df.head(10))
print(f"Melted dataframe shape: {df.shape}")

Melting data...


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
5,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
6,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


Melted dataframe shape: (5918109, 8)


### Merging with Calendar and Prices

In [20]:
print("Merging with calendar...")
df = pd.merge(df, calendar, on='d', how='left')

print("Merging with prices...")
df = pd.merge(df, prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# Drop rows where price is NaN (meaning the product was not being sold yet)
df.dropna(subset=['sell_price'], inplace=True)

df = reduce_mem_usage(df)
display(df.head(10))

Merging with calendar...
Merging with prices...
Memory usage of dataframe is 570.81 MB
Memory usage after optimization is: 570.81 MB
Decreased by 0.0%


/home/anderson/.pyenv/versions/demand-forecast-xai-env/lib/python3.12/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.459961
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,1.559570
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,3.169922
11,HOBBIES_1_012_CA_1_evaluation,HOBBIES_1_012,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,5.980469
14,HOBBIES_1_015_CA_1_evaluation,HOBBIES_1_015,HOBBIES_1,HOBBIES,CA_1,CA,d_1,4,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.700195
15,HOBBIES_1_016_CA_1_evaluation,HOBBIES_1_016,HOBBIES_1,HOBBIES,CA_1,CA,d_1,5,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.700195
21,HOBBIES_1_022_CA_1_evaluation,HOBBIES_1_022,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.859375
22,HOBBIES_1_023_CA_1_evaluation,HOBBIES_1_023,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,3.439453
27,HOBBIES_1_028_CA_1_evaluation,HOBBIES_1_028,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.671875
28,HOBBIES_1_029_CA_1_evaluation,HOBBIES_1_029,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,7.441406


### Defensive Programming (Nullifying the Future)

In [21]:
# The exact date corresponding to day 1913 (end of training data)
SPLIT_DATE = '2016-04-24'

print("Isolating ground truth and nullifying the test window target...")

# Nullify the sales in the main dataframe for any date after the split
df.loc[df['date'] > SPLIT_DATE, 'sales'] = np.nan

# Security Check: Verify that the test window only contains NaN for sales
test_window_sales = df[df['date'] > SPLIT_DATE]['sales'].unique()
print(f"Unique values in 'sales' after {SPLIT_DATE}: {test_window_sales}")

Isolating ground truth and nullifying the test window target...
Unique values in 'sales' after 2016-04-24: [nan]


### Temporal Feature Engineering

In [22]:
print("Creating temporal features...")

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Extracting basic date features
df['day_of_month'] = df['date'].dt.day.astype(np.int8)
df['day_of_week'] = df['date'].dt.dayofweek.astype(np.int8)
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(np.int8)

# Handling categorical events (LightGBM prefers integer categories over strings)
event_cols = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
for col in event_cols:
    df[col] = df[col].astype('category').cat.codes.astype(np.int16)

display(df[['date', 'day_of_month', 'event_name_1', 'snap_CA']].head(10))

Creating temporal features...


,date,day_of_month,event_name_1,snap_CA
7,2011-01-29,29,-1,0
8,2011-01-29,29,-1,0
9,2011-01-29,29,-1,0
11,2011-01-29,29,-1,0
14,2011-01-29,29,-1,0
15,2011-01-29,29,-1,0
21,2011-01-29,29,-1,0
22,2011-01-29,29,-1,0
27,2011-01-29,29,-1,0
28,2011-01-29,29,-1,0


### Lags and Rolling Means (Leakage-Free)

In [23]:
# Sorting is CRITICAL before calculating lags and rolling means
print("Sorting data chronologically...")
df.sort_values(by=['id', 'date'], inplace=True)

print("Calculating Lags...")
lags = [7, 28]
for lag in lags:
    df[f'sales_lag_{lag}'] = df.groupby(['id'])['sales'].shift(lag).astype(np.float16)

print("Calculating Rolling Means (Anchored to Lag 28)...")
windows = [7, 28]
for window in windows:
    df[f'rolling_mean_{window}'] = df.groupby(['id'])['sales_lag_28'].transform(
        lambda x: x.rolling(window).mean()
    ).astype(np.float16)

Sorting data chronologically...
Calculating Lags...
Calculating Rolling Means (Anchored to Lag 28)...


### Validation Check

In [27]:
# Inspect a single product to ensure features are populated correctly in the blind test window
sample_item = df['id'].iloc[0]

cols_to_view = ['id', 'd', 'date', 'sales', 'sales_lag_7', 'sales_lag_28', 'rolling_mean_7']

print(f"Validating test window for item: {sample_item}")
print("Notice how 'sales' is NaN, but the features are successfully calculated from the past:")

# Display the transition from training to testing
display(df[(df['id'] == sample_item) & (df['date'] >= '2016-04-20')][cols_to_view])

Validating test window for item: FOODS_1_001_CA_1_evaluation
Notice how 'sales' is NaN, but the features are successfully calculated from the past:


/home/anderson/.pyenv/versions/demand-forecast-xai-env/lib/python3.12/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,id,d,date,sales,sales_lag_7,sales_lag_28,rolling_mean_7
5819104,FOODS_1_001_CA_1_evaluation,d_1909,2016-04-20,1.0,1.0,2.0,1.571289
5822153,FOODS_1_001_CA_1_evaluation,d_1910,2016-04-21,0.0,1.0,0.0,1.286133
5825202,FOODS_1_001_CA_1_evaluation,d_1911,2016-04-22,1.0,0.0,1.0,1.286133
5828251,FOODS_1_001_CA_1_evaluation,d_1912,2016-04-23,1.0,2.0,1.0,1.142578
5831300,FOODS_1_001_CA_1_evaluation,d_1913,2016-04-24,0.0,0.0,0.0,0.856934
5834349,FOODS_1_001_CA_1_evaluation,d_1914,2016-04-25,NaN,4.0,2.0,1.142578
5837398,FOODS_1_001_CA_1_evaluation,d_1915,2016-04-26,NaN,1.0,1.0,1.000000
5840447,FOODS_1_001_CA_1_evaluation,d_1916,2016-04-27,NaN,1.0,1.0,0.856934
5843496,FOODS_1_001_CA_1_evaluation,d_1917,2016-04-28,NaN,0.0,0.0,0.856934
5846545,FOODS_1_001_CA_1_evaluation,d_1918,2016-04-29,NaN,1.0,4.0,1.286133
